### Zara Zackigs Zurückkehr – Lösung Schritt für Schritt

*40. Bundeswettbewerb Informatik (2021/2022), 2. Runde, Bonusaufgabe*

Zara Zackig hat inzwischen große Karriere gemacht und ihre eigene Informatikfirma aufgebaut, wobei sie ihre Leidenschaft zur
Kryptographie erfolgreich einsetzen konnte. Diese beschäftigt sie auch privat, was ihr folgendes Problem bereitete:

Zara besitzt jetzt zehn Ferienhäuser, die sie zyklisch jedes Wochenende aufsucht. Zur Zugangskontrolle verwendet sie ein selbstentworfenes
System, das Lochkarten mit 128 Positionen verwendet. Jede Lochkarte kann also ein beliebiges 128 Bits umfassendes Codewort darstellen.
Für ihre zehn Häuser stellte sie zehn Karten mit zufällig generierten Codewörtern her, sortierte sie anschließend aufsteigend als
w1, …, w10 und codierte die Schlösser der zehn Häuser, wobei Haus k das Codewort wk erhielt.

Nach einiger Zeit bekam Zara aber Angst, dass eine Karte auf dem Weg verloren gehen könnte. Die meisten Menschen hätten jetzt wahrscheinlich
einfach Kopien der Karten an einem sicheren Platz hinterlegt. Nicht so Zara! Sie stellte nur eine zusätzliche Karte her, deren Bitmuster das
exklusive Oder aller zehn Karten enthält. Mithilfe dieser Sicherungskarte kann sie jetzt im Notfall eine verlorene Karte rekonstruieren.

So weit so gut. Leider sind Zaras Freunde ebenso verrückt wie sie selbst und spielten ihr einen Streich: Sie erstellten weitere 100 Karten
mit zufälligen Codewörtern und mischten sie mit Zaras elf Karten in einem großen Stapel, den dann Zara so vorfand. Dummerweise sind die
elektronischen Schlösser in Zaras Häusern so konstruiert, dass sie nach drei fehlerhaften Zugangsversuchen das Haus für 24 Stunden verriegeln.
Zara konnte das Dilemma allerdings lösen, indem sie ein Programm schrieb, das die 111 Karten einlas und dann herausfand, welche 11 Karten
die echten waren.

#### Aufgabe

a) Tu es Zara gleich und schreibe ein Programm, das diese Aufgabenstellung bewältigt. Wende es auf die 111 Karten an, die du auf den
BWINF-Webseiten findest. Welche Karten sind die richtigen und welche wurden durch die „Freunde“ hinzugefügt?

b) Wie kann nun Zara mithilfe der 11 gefundenen Karten am nächsten Wochenende das nächste Haus aufsperren, ohne dafür mehr als zwei
Fehlversuche zu benötigen?

c) Auf den BWINF-Webseiten findest du weitere Beispielaufgaben, an denen du dein Können demonstrieren kannst. Einige davon sind sehr schwer!

#### Beispieldaten

`beispieldaten/stapel0.txt` bis `stapel5.txt`. Die erste Zeile enthält `N K M`: Zahl der Karten, Zahl der Schlüsselkarten (ohne
Sicherungskarte) und Zahl der Bits je Karte. Danach folgt je Karte eine Zeile mit `M` Zeichen `0`/`1`. (Eine Formatbeschreibung gibt es nicht;
wir haben das aus den Dateien abgelesen. `stapel2` ist die Aufgabe aus dem Text: `111 10 128`.)

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **XOR (exklusives Oder):** `a ^ b` rechnet bitweise: gleiche Bits ergeben 0, verschiedene 1. Es gilt `a ^ a = 0`. Daraus folgt: Die elf
  echten Karten haben zusammen das XOR `0` (die Sicherungskarte ist das XOR der zehn anderen). Und es ist egal, welche der elf die
  Sicherungskarte ist: Jede ist das XOR der übrigen.
- **Die Aufgabe heißt damit:** Finde `K + 1` Karten, deren XOR `0` ist.
- **Naiv** probiert man alle Auswahlen durch. Für `N = 111` und `K + 1 = 11` sind das mehrere hundert Billionen, viel zu viele.
- **Lineare Algebra über {0, 1} (Gauß-Verfahren):** Alle Auswahlen mit XOR 0 entstehen aus wenigen „Grundabhängigkeiten“ durch
  XOR. Das spart fast die ganze Suche.
- **Teile und herrsche (meet in the middle):** Wenn `K + 1` klein ist, speichert man alle XOR-Werte kleiner Auswahlen in einer Tabelle
  (Dictionary) und sucht zu jeder zweiten Hälfte den Partner.

**Bibliotheken und Funktionen**
- Python-Ganzzahlen haben beliebig viele Bits: Eine Karte mit 128 Bits ist einfach eine `int`, `int('0110…', 2)` liest sie ein, `^` ist XOR.
- `bin(x).count('1')` zählt die Einsen (die Lochungen) einer Zahl, `x.bit_length()` gibt die Stelle des höchsten Bits.
- `itertools.combinations(range(n), j)` liefert alle Auswahlen von `j` aus `n` Elementen, `math.comb(n, j)` zählt sie.
- `dict.setdefault(schluessel, [])` legt beim ersten Mal eine leere Liste an.

## Die Karten einlesen

Eine Karte wird als `int` gespeichert, ihre Bits sind die Löcher. Die Funktion `lies` gibt `N`, `K`, `M` und die Liste der Karten zurück.
`karte_text` macht aus einer Zahl wieder die `0`/`1`-Schreibweise.

In [ ]:
import math
import itertools


def lies(name):
    with open('beispieldaten/' + name) as f:
        zeilen = f.read().split('\n')
    n, k, m = map(int, zeilen[0].split())
    karten = [int(z, 2) for z in zeilen[1:n + 1]]
    return n, k, m, karten


def karte_text(karte, m):
    return format(karte, '0%db' % m)


def anzahl_einsen(x):
    return bin(x).count('1')


BEISPIELE = {}
for nummer in range(6):
    BEISPIELE[nummer] = lies('stapel%d.txt' % nummer)
    n, k, m, karten = BEISPIELE[nummer]
    print('stapel%d: N = %3d Karten, K = %2d, M = %3d Bits, Auswahlen von K+1: %d' % (nummer, n, k, m, math.comb(n, k + 1)))

stapel0: N =  20 Karten, K =  4, M =  32 Bits, Auswahlen von K+1: 15504
stapel1: N =  20 Karten, K =  8, M =  32 Bits, Auswahlen von K+1: 167960
stapel2: N = 111 Karten, K = 10, M = 128 Bits, Auswahlen von K+1: 473239787751081
stapel3: N = 161 Karten, K = 10, M = 128 Bits, Auswahlen von K+1: 33283806441690896
stapel4: N = 181 Karten, K = 10, M = 128 Bits, Auswahlen von K+1: 125519813934742980
stapel5: N = 200 Karten, K =  4, M =  64 Bits, Auswahlen von K+1: 2535650040


So viele Auswahlen gäbe es bei Durchprobieren. Bei `stapel2` (die Aufgabe aus dem Text) sind es 473 239 787 751 081, das ist nicht zu schaffen.
Die ersten beiden Stapel sind mit rund 15 000 bzw. 170 000 Auswahlen dagegen noch gut durchprobierbar.

## Schritt 1: Durchprobieren für die kleinen Stapel

Zuerst die einfachste Lösung: Alle Auswahlen von `K + 1` Karten bilden, XOR bilden, auf `0` prüfen.

In [ ]:
def xor_von(karten, indizes):
    x = 0
    for i in indizes:
        x ^= karten[i]
    return x


def suche_naiv(karten, k1):
    """Alle Auswahlen von k1 Karten mit XOR 0 (als Liste von Index-Tupeln)."""
    treffer = []
    for auswahl in itertools.combinations(range(len(karten)), k1):
        if xor_von(karten, auswahl) == 0:
            treffer.append(auswahl)
    return treffer


for nummer in (0, 1):
    n, k, m, karten = BEISPIELE[nummer]
    print('stapel%d:' % nummer, suche_naiv(karten, k + 1))

stapel0: [(2, 3, 5, 9, 11)]
stapel1: [(1, 2, 4, 6, 7, 9, 11, 14, 15)]


Bei `stapel0` sind es die Karten 2, 3, 5, 9, 11 (gezählt ab 0 in der Reihenfolge der Datei) und bei `stapel1` die Karten
1, 2, 4, 6, 7, 9, 11, 14, 15. Das stimmt mit den Tabellen der Lösungshinweise überein. Es gibt jeweils genau *eine* Auswahl.
Für `stapel2` bis `stapel5` brauchen wir etwas Besseres.

## Schritt 2: Gauß-Verfahren – die „Grundabhängigkeiten“ der Karten

**Idee.** Wir gehen die Karten der Reihe nach durch und halten eine Liste von Karten, die wir schon „verbaut“ haben, jede mit einem
eigenen höchsten Bit. Eine neue Karte reduzieren wir: Hat sie das gleiche höchste Bit wie eine verbaute Karte, wird diese per XOR
abgezogen. Das wiederholen wir, bis die Karte entweder ein neues höchstes Bit hat (dann wird sie verbaut) oder zu `0` wird.

Wir merken uns zu jeder Karte, *aus welchen Originalkarten* sie per XOR entstanden ist, in einer Maske (Bit `i` der Maske heißt:
Originalkarte `i` ist dabei). Wird eine Karte zu `0`, dann ist ihre Maske eine Auswahl von Originalkarten mit XOR `0`, eine
**Abhängigkeit**.

Jede Auswahl mit XOR `0` ist ein XOR solcher Abhängigkeiten. Mehr als `N − M` Grundabhängigkeiten gibt es nicht, denn mit `M` Bits
können höchstens `M` Karten unabhängig sein. Bei `stapel2` mit `N = 111 < M = 128` bleibt nur *eine* Abhängigkeit übrig, und das ist
schon die Lösung.

In [ ]:
def abhaengigkeiten(karten):
    """Masken (Bit i = Karte i) von Kartenauswahlen mit XOR 0, aus denen sich alle solchen Auswahlen durch XOR bilden lassen."""
    verbaut = {}          # höchstes Bit -> (reduzierte Karte, Maske der Originalkarten)
    ergebnis = []
    for i, karte in enumerate(karten):
        maske = 1 << i
        while karte != 0:
            h = karte.bit_length() - 1
            if h in verbaut:
                karte2, maske2 = verbaut[h]
                karte ^= karte2
                maske ^= maske2
            else:
                verbaut[h] = (karte, maske)
                break
        if karte == 0:
            ergebnis.append(maske)
    return ergebnis


def masken_karten(maske):
    return [i for i in range(maske.bit_length()) if maske >> i & 1]


for nummer in range(6):
    n, k, m, karten = BEISPIELE[nummer]
    abh = abhaengigkeiten(karten)
    for maske in abh:                                    # Probe: jede Abhängigkeit hat XOR 0
        assert xor_von(karten, masken_karten(maske)) == 0
    print('stapel%d: %d Abhängigkeiten (N - M = %d)' % (nummer, len(abh), n - m))

stapel0: 1 Abhängigkeiten (N - M = -12)
stapel1: 1 Abhängigkeiten (N - M = -12)
stapel2: 1 Abhängigkeiten (N - M = -17)
stapel3: 33 Abhängigkeiten (N - M = 33)
stapel4: 53 Abhängigkeiten (N - M = 53)
stapel5: 136 Abhängigkeiten (N - M = 136)


Bei `stapel0` bis `stapel2` (`N ≤ M`, zufällige Karten sind dann fast sicher unabhängig) bleibt je eine einzige Abhängigkeit, die der echten Karten.
Bei `stapel3` bis `stapel5` (`N > M`) sind es genau `N − M` Abhängigkeiten, von denen die echte Auswahl nur eine Kombination ist.

**Die echte Auswahl finden.** Jede Auswahl mit XOR 0 ist ein XOR einiger Abhängigkeiten. Damit wir die möglichen XOR-Kombinationen
übersichtlich abzählen können, machen wir die Abhängigkeiten „schön“: Jede bekommt eine **eigene Pivotkarte** (ihr höchstes Bit), die in keiner anderen
Abhängigkeit vorkommt. (Die höchsten Bits der Abhängigkeiten sind schon verschieden; es genügt, bei jeder neuen Abhängigkeit die Pivotkarten der
früheren per XOR zu entfernen.) Wählt man dann `j` Abhängigkeiten aus, enthält das Ergebnis mindestens diese `j` Pivotkarten. Wir
müssen deshalb nur Kombinationen mit höchstens `K + 1` Abhängigkeiten ansehen, und zwar zuerst mit `j = 1`, dann `j = 2` usw.: Die echte
Auswahl enthält grob geschätzt nur etwa `(K+1)·d/N` der `d` Pivotkarten, also wenige.

In [ ]:
def schoene_basis(abh):
    """Jede Maske bekommt als Pivot ihr höchstes Bit; kein früherer Pivot kommt in einer späteren Maske vor."""
    pivots, basis = [], []
    for maske in abh:
        for p, q in zip(pivots, basis):
            if maske >> p & 1:
                maske ^= q
        pivots.append(maske.bit_length() - 1)
        basis.append(maske)
    return basis


def suche_gauss(karten, k1):
    """Findet eine Auswahl von k1 Karten mit XOR 0. Gibt (Karten-Indizes, Zahl der geprüften Kombinationen, Zahl d der Basisvektoren)."""
    basis = schoene_basis(abhaengigkeiten(karten))
    d = len(basis)
    geprueft = 0
    for j in range(1, min(d, k1) + 1):
        for auswahl in itertools.combinations(range(d), j):
            geprueft += 1
            maske = 0
            for i in auswahl:
                maske ^= basis[i]
            if anzahl_einsen(maske) == k1:
                return masken_karten(maske), geprueft, d
    return None, geprueft, d


for nummer in range(5):
    n, k, m, karten = BEISPIELE[nummer]
    karten_idx, geprueft, d = suche_gauss(karten, k + 1)
    assert xor_von(karten, karten_idx) == 0 and len(karten_idx) == k + 1
    print('stapel%d: d = %2d, %6d Kombinationen geprüft -> Karten %s' % (nummer, d, geprueft, karten_idx))

stapel0: d =  1,      1 Kombinationen geprüft -> Karten [2, 3, 5, 9, 11]
stapel1: d =  1,      1 Kombinationen geprüft -> Karten [1, 2, 4, 6, 7, 9, 11, 14, 15]
stapel2: d =  1,      1 Kombinationen geprüft -> Karten [9, 20, 26, 53, 57, 71, 76, 78, 89, 95, 99]
stapel3: d = 33,    690 Kombinationen geprüft -> Karten [23, 26, 34, 43, 71, 98, 110, 120, 127, 133, 144]
stapel4: d = 53, 268476 Kombinationen geprüft -> Karten [0, 23, 76, 83, 91, 95, 116, 146, 154, 159, 160]


Damit sind `stapel0` bis `stapel4` gelöst. Gegenüber dem Durchprobieren (`stapel2`: 473 Billionen Auswahlen) haben wir bei `stapel2` nur noch eine
Kombination und bei `stapel3` 690 und bei `stapel4` knapp 270 000 geprüft.

## Schritt 3: Teile und herrsche – für `stapel5`

Bei `stapel5` gilt `N = 200`, `M = 64`, also `d = N − M = 136` Abhängigkeiten, aber nur `K + 1 = 5` echte Karten. Das Gauß-Verfahren
hat nun viel zu viele Kombinationen: Die echte Auswahl besteht aus fünf Karten, die alle zu den „freien“ Karten gehören, es müssten also bis zu
`j = 5` der 136 Basisvektoren kombiniert werden.

In [ ]:
d5 = 200 - 64
print('Kombinationen bis j = 4:', sum(math.comb(d5, j) for j in range(1, 5)))
print('Kombinationen bis j = 5:', sum(math.comb(d5, j) for j in range(1, 6)))

Kombinationen bis j = 4: 14053186
Kombinationen bis j = 5: 373986298


Mehrere hundert Millionen Kombinationen sind für Python zu viel. Aber `K + 1` ist hier klein. Wir teilen die gesuchte Auswahl von 5 Karten in
2 + 3 Karten. Eine Auswahl hat XOR 0, wenn das XOR der ersten Hälfte gleich dem XOR der zweiten Hälfte ist (denn `a ^ a = 0`).
Also:

1. Berechne das XOR für alle Paare von Karten und lege es in ein Dictionary (`XOR → Liste der Paare`).
2. Gehe alle Dreierauswahlen durch und schaue nach, ob ihr XOR im Dictionary steht, mit einem Paar, das keine Karte gemeinsam hat.

Das sind nur `C(200, 2) = 19 900` Einträge und höchstens `C(200, 3) = 1 313 400` Dreierauswahlen.

In [ ]:
def suche_teile(karten, k1):
    """Findet k1 Karten mit XOR 0: Auswahlen der Größe a in einem Dictionary, dann Auswahlen der Größe b = k1 - a suchen.
    Gibt (Karten-Indizes, Zahl der geprüften Auswahlen der zweiten Hälfte)."""
    a = k1 // 2
    b = k1 - a
    tabelle = {}
    for auswahl in itertools.combinations(range(len(karten)), a):
        tabelle.setdefault(xor_von(karten, auswahl), []).append(auswahl)
    geprueft = 0
    for auswahl in itertools.combinations(range(len(karten)), b):
        geprueft += 1
        for partner in tabelle.get(xor_von(karten, auswahl), []):
            if not set(partner) & set(auswahl):
                return sorted(partner + auswahl), geprueft
    return None, geprueft


for nummer in (0, 1, 5):
    n, k, m, karten = BEISPIELE[nummer]
    karten_idx, geprueft = suche_teile(karten, k + 1)
    assert xor_von(karten, karten_idx) == 0 and len(karten_idx) == k + 1
    print('stapel%d: %6d Auswahlen der zweiten Hälfte geprüft -> Karten %s' % (nummer, geprueft, karten_idx))

stapel0:    326 Auswahlen der zweiten Hälfte geprüft -> Karten [2, 3, 5, 9, 11]
stapel1:   4011 Auswahlen der zweiten Hälfte geprüft -> Karten [1, 2, 4, 6, 7, 9, 11, 14, 15]
stapel5: 956479 Auswahlen der zweiten Hälfte geprüft -> Karten [70, 77, 163, 167, 185]


`stapel5` ist damit gelöst (Karten 70, 77, 163, 167, 185), nach knapp einer Million Dreierauswahlen. Die kleinen Stapel `stapel0` und
`stapel1` liefern mit beiden Verfahren dieselben Karten wie das Durchprobieren. Für `stapel2` bis `stapel4` wäre Teile und herrsche
dagegen viel zu langsam: Die Tabelle hätte bei `K + 1 = 11` Karten Einträge für alle Fünferauswahlen, und das sind schon bei `stapel2` über 128 Millionen und bei `stapel4` über 1,5 Milliarden.

In [ ]:
print('stapel2: Fünferauswahlen von 111 Karten:', math.comb(111, 5))
print('stapel4: Fünferauswahlen von 181 Karten:', math.comb(181, 5))

stapel2: Fünferauswahlen von 111 Karten: 128164707
stapel4: Fünferauswahlen von 181 Karten: 1531144341


## Schritt 4: Das Verfahren wählen und alle Beispiele lösen

Welches Verfahren günstiger ist, schätzen wir grob ab: Beim Gauß-Verfahren rechnen wir mit Kombinationen bis `j ≈ (K+1)·d/N + 1`, bei
Teile und herrsche mit `C(N, ⌈(K+1)/2⌉)` Auswahlen. Das kleinere nehmen wir. Die Hinweise nennen für `stapel0` bis `stapel5` die Indizes der
echten Karten (gezählt ab 0); wir vergleichen damit.

In [ ]:
def kosten_gauss(n, m, k1):
    d = max(n - m, 1)
    j = min(d, round(k1 * d / n) + 1)
    return sum(math.comb(d, i) for i in range(1, j + 1))


def kosten_teile(n, k1):
    return math.comb(n, (k1 + 1) // 2)


def loese(n, k, m, karten):
    """Gibt (Karten-Indizes, Verfahren, geprüfte Kombinationen)."""
    if kosten_gauss(n, m, k + 1) <= kosten_teile(n, k + 1):
        idx, geprueft, d = suche_gauss(karten, k + 1)
        return idx, 'Gauß', geprueft
    idx, geprueft = suche_teile(karten, k + 1)
    return idx, 'Teile', geprueft


HINWEISE = {
    0: {2, 9, 11, 5, 3},
    1: {15, 1, 11, 4, 7, 14, 2, 6, 9},
    2: {57, 20, 89, 9, 95, 53, 26, 76, 71, 78, 99},
    3: {133, 43, 120, 127, 98, 71, 23, 26, 110, 144, 34},
    4: {76, 23, 91, 83, 95, 116, 159, 154, 146, 0, 160},
    5: {167, 70, 185, 163, 77},
}
LOESUNGEN = {}
for nummer in range(6):
    n, k, m, karten = BEISPIELE[nummer]
    idx, verfahren, geprueft = loese(n, k, m, karten)
    LOESUNGEN[nummer] = idx
    print('stapel%d (%s, %d Kombinationen), wie in den Hinweisen: %s' % (nummer, verfahren, geprueft, set(idx) == HINWEISE[nummer]))
    print('   echte Karten:', idx)

stapel0 (Gauß, 1 Kombinationen), wie in den Hinweisen: True
   echte Karten: [2, 3, 5, 9, 11]
stapel1 (Gauß, 1 Kombinationen), wie in den Hinweisen: True
   echte Karten: [1, 2, 4, 6, 7, 9, 11, 14, 15]
stapel2 (Gauß, 1 Kombinationen), wie in den Hinweisen: True
   echte Karten: [9, 20, 26, 53, 57, 71, 76, 78, 89, 95, 99]
stapel3 (Gauß, 690 Kombinationen), wie in den Hinweisen: True
   echte Karten: [23, 26, 34, 43, 71, 98, 110, 120, 127, 133, 144]
stapel4 (Gauß, 268476 Kombinationen), wie in den Hinweisen: True
   echte Karten: [0, 23, 76, 83, 91, 95, 116, 146, 154, 159, 160]
stapel5 (Teile, 956479 Kombinationen), wie in den Hinweisen: True
   echte Karten: [70, 77, 163, 167, 185]


Alle sechs Stapel liefern genau die Karten der Lösungshinweise. Die übrigen Karten wurden von den „Freunden“ hinzugefügt.

**Ist die Lösung eindeutig?** Wenn `d = 1` ist (`stapel0`, `stapel1`, `stapel2`), gibt es überhaupt nur eine einzige nichttriviale
Auswahl mit XOR 0, die Lösung ist also sicher eindeutig. Bei den anderen kann es theoretisch eine zweite zufällige Auswahl geben, die genauso viele
Karten hat. Jede der `C(N, K+1) − 1` falschen Auswahlen hat ein zufälliges XOR, das mit Wahrscheinlichkeit `2^−M` gleich `0` ist.

In [ ]:
for nummer in range(6):
    n, k, m, karten = BEISPIELE[nummer]
    wahrscheinlichkeit = -math.expm1((math.comb(n, k + 1) - 1) * math.log1p(-2.0 ** -m))
    print('stapel%d: Wahrscheinlichkeit einer zweiten Auswahl höchstens %.1e' % (nummer, wahrscheinlichkeit))
    assert wahrscheinlichkeit < 1e-4

stapel0: Wahrscheinlichkeit einer zweiten Auswahl höchstens 3.6e-06
stapel1: Wahrscheinlichkeit einer zweiten Auswahl höchstens 3.9e-05
stapel2: Wahrscheinlichkeit einer zweiten Auswahl höchstens 1.4e-24
stapel3: Wahrscheinlichkeit einer zweiten Auswahl höchstens 9.8e-23
stapel4: Wahrscheinlichkeit einer zweiten Auswahl höchstens 3.7e-22
stapel5: Wahrscheinlichkeit einer zweiten Auswahl höchstens 1.4e-10


Das ist überall kleiner als `10^−4` (wie auch die Hinweise sagen). Streng bewiesen ist die Eindeutigkeit also nur für `stapel0` bis `stapel2`.

## Teil b: Das richtige Haus aufsperren

Zara sortiert die elf Karten aufsteigend (als Zahlen, oder auch als Text der Bits). Die zehn echten Schlüssel `w1 < … < w10` stehen
in dieser Reihenfolge, dazwischen steckt an *irgendeiner* Stelle `p` die Sicherungskarte. Sie weiß nicht, an welcher, denn jede der elf
Karten könnte die Sicherungskarte sein.

Für Haus `k` probiert sie die `k`-te Karte der sortierten Liste:

- Steht die Sicherungskarte **hinter** der gesuchten Karte (`p > k`), ist die `k`-te Karte die richtige, der erste Versuch passt.
- Steht sie **davor oder an** Stelle `k` (`p ≤ k`), ist alles um eins nach rechts verschoben, die `k`-te Karte ist falsch, und die
  `(k+1)`-te ist die richtige: **ein** Fehlversuch, dann passt es.

Mehr als ein Fehlversuch kommt also nie vor (erlaubt sind zwei). Wir spielen alle elf möglichen Positionen der Sicherungskarte für `stapel2` durch.

In [ ]:
n, k, m, karten = BEISPIELE[2]
sortiert = sorted(karten[i] for i in LOESUNGEN[2])      # die elf Karten, aufsteigend sortiert
print('Die elf Karten (erste 24 Bits), aufsteigend:')
for nr, karte in enumerate(sortiert, 1):
    print('%2d  %s' % (nr, karte_text(karte, m)[:24]))


def haus_oeffnen(sortiert, haus, p):
    """Haus 1..10, Sicherungskarte an Position p (1..11 in der sortierten Liste).
    Probiert die Karte `haus`, dann die nächste. Gibt die Zahl der Fehlversuche zurück."""
    echte = sortiert[:p - 1] + sortiert[p:]            # die zehn Hausschlüssel w1..w10
    code = echte[haus - 1]
    fehler = 0
    for position in (haus - 1, haus):                   # Karte haus, dann Karte haus + 1 (Positionen ab 0)
        if sortiert[position] == code:
            return fehler
        fehler += 1
    raise ValueError('nicht gefunden')


groesster = 0
for p in range(1, 12):
    fehlversuche = [haus_oeffnen(sortiert, haus, p) for haus in range(1, 11)]
    groesster = max(groesster, max(fehlversuche))
    print('Sicherungskarte an Position %2d: Fehlversuche je Haus 1..10: %s' % (p, fehlversuche))
print('Höchstens', groesster, 'Fehlversuch(e) bei jedem Haus')

Die elf Karten (erste 24 Bits), aufsteigend:
 1  001010000110000100101110
 2  001010111110001010110101
 3  011010010010110001010011
 4  011010111010001101110100
 5  011101100111100011100111
 6  100000000001001001100110
 7  101010110000011011000001
 8  101011111100100100101001
 9  110000110001001101110001
10  110111100001010011011111
11  111011101010111001111011
Sicherungskarte an Position  1: Fehlversuche je Haus 1..10: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Sicherungskarte an Position  2: Fehlversuche je Haus 1..10: [0, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Sicherungskarte an Position  3: Fehlversuche je Haus 1..10: [0, 0, 1, 1, 1, 1, 1, 1, 1, 1]
Sicherungskarte an Position  4: Fehlversuche je Haus 1..10: [0, 0, 0, 1, 1, 1, 1, 1, 1, 1]
Sicherungskarte an Position  5: Fehlversuche je Haus 1..10: [0, 0, 0, 0, 1, 1, 1, 1, 1, 1]
Sicherungskarte an Position  6: Fehlversuche je Haus 1..10: [0, 0, 0, 0, 0, 1, 1, 1, 1, 1]
Sicherungskarte an Position  7: Fehlversuche je Haus 1..10: [0, 0, 0, 0, 0, 0, 1, 1, 1, 1]

Bei Sicherungskarte an Position `p` sind es genau die Häuser `k ≥ p`, bei denen ein Fehlversuch auftritt. Zara braucht also nie mehr als
einen Fehlversuch pro Haus, und die Sperre nach drei Fehlversuchen kommt nie ins Spiel.

## Zum Schluss: Geht das auch mit MiniZinc?

Als Modell bietet sich an: Für jede Karte `i` eine 0/1-Variable `x[i]` („Karte gehört zur Auswahl“), genau `K + 1` davon sind 1, und an jeder
Bitposition `b` muss die Zahl der gewählten Karten, die dort eine `1` haben, **gerade** sein (XOR `0`). Gerade heißt: Die Summe ist `2·t[b]`
mit einer Hilfsvariablen `t[b]`. Das ist in den Arbeitsblättern bekannt (`sum … where`, `forall`, 2D-Arrays).
```minizinc
int: N;                                   % Zahl der Karten
int: M;                                   % Bits je Karte
int: K1;                                  % gesuchte Zahl von Karten (K + 1)
array[1..M, 1..N] of 0..1: a;             % a[b, i] = Bit b der Karte i
array[1..N] of var 0..1: x;               % x[i] = 1: Karte i gehört zur Auswahl
array[1..M] of var 0..N div 2: t;         % Hilfsvariable für die Parität

constraint sum(x) = K1;
% an jeder Bitposition ist die Zahl der gewählten Einsen gerade (XOR = 0)
constraint forall(b in 1..M)(sum(i in 1..N where a[b, i] = 1)(x[i]) = 2 * t[b]);

solve satisfy;
output [show(i - 1) ++ " " | i in 1..N where fix(x[i]) = 1];
```

Die Datendatei erzeugt die Python-Zelle `als_dzn`. Ein winziges Beispiel (drei Karten mit je vier Bits, die XOR 0 haben):

In [ ]:
def als_dzn(n, k, m, karten):
    werte = []
    for b in range(m):                                  # Zeile b: Bit b (von links) aller Karten
        werte += [str(karte >> (m - 1 - b) & 1) for karte in karten]
    return 'N = %d;\nM = %d;\nK1 = %d;\na = array2d(1..%d, 1..%d, [%s]);' % (n, m, k + 1, m, n, ', '.join(werte))


print(als_dzn(3, 2, 4, [int('0110', 2), int('1010', 2), int('1100', 2)]))

N = 3;
M = 4;
K1 = 3;
a = array2d(1..4, 1..3, [0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0]);


Ergebnis beim Erzeugen dieses Notebooks (Zeitlimit 10 Sekunden je Aufruf; Solver in MiniZinc über *Solver configuration* gewählt). „gelöst“ heißt:
Der Solver findet die Auswahl der Hinweise, „?“: Zeitlimit.

| Stapel | N | K + 1 | M | Gecode | Chuffed | CP-SAT |
|--------|---|-------|---|--------|---------|--------|
| stapel0 | 20 | 5 | 32 | gelöst | gelöst | gelöst |
| stapel1 | 20 | 9 | 32 | gelöst | gelöst | gelöst |
| stapel2 | 111 | 11 | 128 | ? | ? | ? |
| stapel3 | 161 | 11 | 128 | ? | ? | ? |
| stapel4 | 181 | 11 | 128 | ? | ? | ? |
| stapel5 | 200 | 5 | 64 | ? | ? | ? |

MiniZinc löst nur die beiden kleinen Stapel, bei denen auch das Durchprobieren geht. Bei den größeren Stapeln scheitern alle Solver. Das hat einen
Grund: Die Bedingungen sind *Paritätsbedingungen* (XOR), und die kann ein Solver, der mit Wertebereichen und Zählen arbeitet, schlecht ausnutzen. Er
probiert Belegungen von `x` und merkt erst spät, dass die Summen nicht gerade sind. Das Gauß-Verfahren nutzt genau diese Struktur
(lineares Gleichungssystem über {0, 1}), die wir dem Solver nicht beibringen. Für diese Aufgabe ist selbstgeschriebener
Python-Code deutlich besser als MiniZinc.

## Für Neugierige

- **Warum sind die Lösungen praktisch eindeutig?** Siehe die Wahrscheinlichkeit oben: Es gibt viele Auswahlen, aber jede trifft die `0` nur mit
  Wahrscheinlichkeit `2^−M`. Mit `M = 128` ist das winzig.
- **Wie schnell ist das Gauß-Verfahren?** Die Reduktion braucht etwa `N · M` Schritte mit `M`-Bit-Zahlen (`O(N·M²)` Bitoperationen). Danach hängt alles
  an der Zahl `d = N − M` der freien Karten und an `K + 1`. In C++ oder Java nimmt man dafür Bit-Arrays aus 64-Bit-Zahlen; Python-`int` tut das für uns.
- **Teile und herrsche ist besser bei kleinem `K` und großem `N − M`** (`stapel5`), Gauß bei großem `K` und kleinem `N − M` (`stapel3`, `stapel4`).
  Eine Mischform: Beim Gauß-Verfahren zuerst die Kombinationen mit wenigen Basisvektoren versuchen (so machen wir es).
- **Weitere Eingaben:** Die Verfahren gelten für beliebige `N`, `M` und `K` (auch `N < M`), `loese` wählt je nach Größe das günstigere.
  Wenn *keine* Auswahl existiert, geben `suche_gauss` bzw. `suche_teile` `None` zurück.